# Phase 3 – Hybrid Search & Re-ranking Notebook

Experiments with:
- BM25 sparse retrieval
- Hybrid (BM25 + dense embedding) retrieval
- Cross-encoder re-ranking
- FAISS vs Qdrant comparison


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import time

In [ ]:
from src.ingestion.chunk_text import load_chunks
from src.embeddings.embed import load_embeddings

chunks   = load_chunks()
embedded = load_embeddings()
print(f'Chunks: {len(chunks)}   Embeddings: {len(embedded)}')

## FAISS vs Qdrant Speed Comparison

In [ ]:
from src.embeddings.faiss_store import load_faiss_index, search_faiss
from src.embeddings.qdrant_store import get_qdrant_client, search_qdrant
from src.retrieval.retriever import embed_query

# Load both stores
faiss_index, faiss_meta = load_faiss_index()

try:
    qdrant_client = get_qdrant_client()
    qdrant_available = True
except:
    qdrant_available = False
    print('⚠️  Qdrant not available. Start Docker first.')

test_queries = [
    'What was IFC net income in 2024?',
    'How much did IFC commit to climate investments?',
    'What is the breakdown of IFC loan portfolio by region?',
]

print('Testing retrieval speed...')
results_data = []

for query in test_queries:
    q_emb = embed_query(query)
    
    # FAISS timing
    t0 = time.perf_counter()
    faiss_results = search_faiss(q_emb, faiss_index, faiss_meta, top_k=5)
    faiss_time = (time.perf_counter() - t0) * 1000
    
    row = {
        'query'     : query[:50],
        'faiss_ms'  : round(faiss_time, 2),
        'faiss_top1': faiss_results[0]['page_number'] if faiss_results else '?',
    }
    
    if qdrant_available:
        t0 = time.perf_counter()
        qdrant_results = search_qdrant(q_emb, qdrant_client, top_k=5)
        qdrant_time = (time.perf_counter() - t0) * 1000
        row['qdrant_ms']   = round(qdrant_time, 2)
        row['qdrant_top1'] = qdrant_results[0]['page_number'] if qdrant_results else '?'
    
    results_data.append(row)

df_speed = pd.DataFrame(results_data)
df_speed

In [ ]:
# Plot speed comparison
if qdrant_available:
    x = range(len(test_queries))
    labels = [q[:30] + '...' for q in test_queries]
    
    plt.figure(figsize=(10, 4))
    plt.bar([i - 0.2 for i in x], df_speed['faiss_ms'],  0.4, label='FAISS',  color='#2196F3')
    plt.bar([i + 0.2 for i in x], df_speed['qdrant_ms'], 0.4, label='Qdrant', color='#FF5722')
    plt.xticks(list(x), labels, rotation=15, ha='right')
    plt.ylabel('Time (ms)')
    plt.title('FAISS vs Qdrant – Search Latency')
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    print('\nFAISS is in-memory → faster for small datasets')
    print('Qdrant has overhead from HTTP → but scales better and supports filtering')

## BM25 Sparse Retrieval

In [ ]:
from src.retrieval.retriever import build_bm25_index, search_bm25

# Build BM25 index on text chunks
build_bm25_index(chunks)

query = 'net income 2024'
bm25_results = search_bm25(query, top_k=5)

print(f'BM25 results for: "{query}"\n')
for i, r in enumerate(bm25_results):
    print(f'{i+1}. Page {r["page_number"]:3d} | Score {r["score"]:.3f} | {r["text"][:120]}...')

## Cross-encoder Re-ranking

In [ ]:
from src.retrieval.retriever import rerank

query = 'What was IFC net income in 2024?'
q_emb = embed_query(query)

# Get top-10 from FAISS (more than we need, so reranker has room to work)
initial_results = search_faiss(q_emb, faiss_index, faiss_meta, top_k=10)

print('=== Before re-ranking (FAISS order) ===')
for i, r in enumerate(initial_results[:5]):
    print(f'{i+1}. Page {r["page_number"]:3d} | Score {r["score"]:.3f} | {r["text"][:80]}...')

print('\n=== After cross-encoder re-ranking ===')
reranked = rerank(query, initial_results, top_k=5)
for i, r in enumerate(reranked):
    print(f'{i+1}. Page {r["page_number"]:3d} | Score {r["rerank_score"]:.3f} | {r["text"][:80]}...')

## Hybrid Retrieval: Dense + BM25 Combined

In [ ]:
from src.retrieval.retriever import Retriever

query = 'What are IFC total assets for fiscal year 2024?'

results_comparison = {}

for mode, rerank_flag in [
    ('faiss',  False),
    ('hybrid', False),
    ('hybrid', True),
]:
    label = f'{mode}' + ('+rerank' if rerank_flag else '')
    try:
        r = Retriever(mode=mode).load(chunks=chunks)
        res = r.retrieve(query, top_k=5, use_rerank=rerank_flag)
        results_comparison[label] = [f"p{x['page_number']}" for x in res]
    except Exception as e:
        results_comparison[label] = [f'Error: {e}']

print(f'Query: {query}\n')
for label, pages in results_comparison.items():
    print(f'  [{label:<18}]  Pages: {pages}')

## Metadata Filtering with Qdrant

In [ ]:
if qdrant_available:
    query = 'What are the financial highlights?'
    q_emb = embed_query(query)
    
    # Without filter
    results_all = search_qdrant(q_emb, qdrant_client, top_k=5)
    print('Without filter (all pages):')
    print([r['page_number'] for r in results_all])
    
    # With page filter: only search pages 1-10
    results_filtered = search_qdrant(q_emb, qdrant_client, top_k=5, page_range=(1, 10))
    print('\nWith page filter (pages 1-10 only):')
    print([r['page_number'] for r in results_filtered])
    
    print('\n✅ Qdrant filtering works! FAISS cannot do this without loading all metadata.')
else:
    print('Start Qdrant with Docker to test filtering.')